# Constraint Satisfaction Matcher

The ConstraintSatisfactionMatcher can be used to optimize any linear function of the baseline covariates. We support constraints on the size of the subset populations and the allowed mismatch.

Here, we demonstrate the optimization of balance subject to size constraints only. Namely, we solve:

\begin{equation}
\begin{aligned}
& \underset{\hat{P}}{\text{minimize}}
& & \sum_k |\mu_{\hat{P}k} - \mu_{Tk}| \\
& \text{subject to}
& & |\hat{P}| = P^* \\
& & & |\hat{T}| = T^* \\
\end{aligned}
\end{equation}

where $P$ and $T$ refer to two populations we are trying to match, $\hat{P}$ and $\hat{T}$ are the subsets of $P$ and $T$ we are seeking, $P^*$ and $T^*$ are fixed integers, and  $k$ indexes the covariates of $P$ and $T$.

In [1]:
import logging 
logging.basicConfig(
    format="%(levelname)-4s [%(filename)s:%(lineno)d] %(message)s",
    level='INFO',
)
from pybalance.utils import (
    BetaBalance, 
    BetaXBalance, 
    GammaBalance, 
    GammaXBalance,
    GammaXTreeBalance
)
from pybalance.sim import generate_toy_dataset
from pybalance.lp import ConstraintSatisfactionMatcher
from pybalance.visualization import (
    plot_numeric_features, 
    plot_categoric_features, 
    plot_binary_features,
    plot_per_feature_loss,
)

In [2]:
time_limit = 60

In [3]:
m = generate_toy_dataset()
m

,age,height,weight,gender,haircolor,country,population,binary_0,binary_1,binary_2,binary_3,patient_id
0,62.511573,190.229250,105.165097,0.0,2,3,pool,0,0,0,0,0
1,68.505065,161.121236,95.001474,0.0,1,1,pool,1,0,1,0,1
2,50.071384,162.325356,84.290576,1.0,0,5,pool,0,0,1,1,2
3,44.423692,150.948096,82.031381,1.0,2,2,pool,0,0,0,1,3
4,41.695052,132.952651,54.857540,0.0,1,3,pool,0,0,1,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...
995,21.474205,168.602546,70.342128,0.0,2,5,target,0,0,0,1,10995
996,40.643320,188.188724,61.611744,0.0,2,4,target,1,0,0,1,10996
997,29.472765,161.408162,57.214095,0.0,0,1,target,0,1,1,1,10997
998,41.291949,150.968833,91.270798,0.0,0,3,target,0,0,0,0,10998


## Optimize Beta (Mean Absolute SMD)

In [4]:
objective = beta = BetaBalance(m)
matcher = matcher_beta = ConstraintSatisfactionMatcher(
    m, 
    time_limit=time_limit,
    objective=objective,
    ps_hinting=False,
    num_workers=4)
matcher.get_params()

INFO [matcher.py:66] Scaling features by factor 240.00 in order to use integer solver with <= 0.2841% loss.


{'objective': 'beta',
 'pool_size': 1000,
 'target_size': 1000,
 'max_mismatch': None,
 'time_limit': 60,
 'num_workers': 4,
 'ps_hinting': False,
 'verbose': True}

In [ ]:
matcher.match()

INFO [matcher.py:475] Solving for match population with pool size = 1000 and target size = 1000, optimizing balance (no max_mismatch cap).
INFO [matcher.py:479] Matching on 15 dimensions ...
INFO [matcher.py:486] Building model variables and constraints ...
INFO [matcher.py:495] Calculating bounds on feature variables ...
INFO [matcher.py:585] Applying size constraints on pool and target ...
INFO [matcher.py:239] Solving with 4 workers ...
INFO [matcher.py:91] Initial balance score: 0.2328
INFO [matcher.py:97] =========================================
INFO [matcher.py:98] Solution 1, time = 1.1 s
INFO [matcher.py:102] Objective:	452948000.0
INFO [matcher.py:123] Balance (beta):	0.2298
INFO [matcher.py:128] Patients (pool):	1000
INFO [matcher.py:130] Patients (target):	1000
INFO [matcher.py:146]  
INFO [matcher.py:97] =========================================
INFO [matcher.py:98] Solution 2, time = 1.6 s
INFO [matcher.py:102] Objective:	452876000.0
INFO [matcher.py:123] Balance (beta):	

Note that it is possible for the "balance" value to go up (hopefully only slightly) as the solver progresses. This behavior is due to rounding errors incurred when casting a continuous problem into the discrete space understood by the solver. The "balance" reported here is the actual balance computed to full accuracy on the original dataset; the "objective" value here is the actual quantity being optimized and should never increase.

In [ ]:
matcher.get_best_match()

In [ ]:
%matplotlib inline

match = matcher.get_best_match()
m_data = m.copy().get_population('pool')
m_data.loc[:, 'population'] = m_data['population'] + ' (prematch)'
match.append(m_data)
# fig = plot_per_feature_loss(match, beta, 'target', debin=False)
fig = plot_numeric_features(match, hue_order=['pool (prematch)', 'pool', 'target', ])
fig = plot_categoric_features(match,  hue_order=['pool (prematch)', 'pool', 'target'])


## Optimize Beta With Cross Terms Added

Sometimes it helps to add a known (non-optimal) solution as a hint to the solver. A natural choice for hinting the solver is to take a solution from PS matching. We can choose to use the PS as a hint to the solver by passing ps_hinting=True.

In [ ]:
objective = beta_x = BetaXBalance(m)
matcher = matcher_betax = ConstraintSatisfactionMatcher(
    m, 
    time_limit=time_limit,
    objective=objective,
    ps_hinting=True,
    num_workers=4)
matcher.get_params()

In [ ]:
matcher.match()

In [ ]:
%matplotlib inline

match = matcher.get_best_match()
m_data = m.copy().get_population('pool')
m_data.loc[:, 'population'] = m_data['population'] + ' (prematch)'
match.append(m_data)
fig = plot_per_feature_loss(match, objective, 'target', debin=False)

## Optimize Gamma (Area Between CDFs)

In [ ]:
objective = gamma = GammaBalance(m)
matcher = matcher_gamma = ConstraintSatisfactionMatcher(
    m, 
    time_limit=time_limit,
    objective=objective,
    ps_hinting=True,
    num_workers=4)
matcher.get_params()

In [ ]:
matcher.match()

In [ ]:
%matplotlib inline

match = matcher.get_best_match()
m_data = m.copy().get_population('pool')
m_data.loc[:, 'population'] = m_data['population'] + ' (prematch)'
match.append(m_data)
fig = plot_per_feature_loss(match, objective, 'target', debin=False)
fig = plot_numeric_features(match, hue_order=['pool (prematch)', 'pool', 'target', ])
fig = plot_categoric_features(match,  hue_order=['pool (prematch)', 'pool', 'target'])